# 01 · Carga, auditoría, limpieza y unión

**Fuente A:** accidentes con víctimas, Ayuntamiento de Madrid (2019-2024).  
**Fuente B:** climatología diaria AEMET (estación Madrid-Retiro).  
**Salida:** `data/processed/accidentes_meteo_madrid.csv`

In [17]:
import sys; sys.path.append("..")
import pandas as pd
from src.config import *
from src.cleaning import *
from src.features import *
pd.set_option("display.max_columns", 60)

## 1. Carga y auditoría de la fuente A (accidentes)

In [18]:
acc_raw = load_accidentes(RAW_ACCIDENTES)
print(acc_raw.shape)
acc_raw.head()

300228-11-accidentes-trafico-detalle-csv.csv: 51,811 filas, 20 columnas
300228-2-accidentes-trafico-detalle-csv.csv: 49,340 filas, 20 columnas
300228-3-accidentes-trafico-detalle-csv.csv: 48,830 filas, 20 columnas
300228-33-accidentes-trafico-detalle-csv.csv: 41,783 filas, 20 columnas
300228-5-accidentes-trafico-detalle-csv.csv: 47,053 filas, 20 columnas
300228-8-accidentes-trafico-detalle-csv.csv: 32,433 filas, 20 columnas
(271250, 20)


,num_expediente,fecha,hora,localizacion,numero,cod_distrito,distrito,tipo_accidente,estado_meteorologico,tipo_vehiculo,tipo_persona,rango_edad,sexo,cod_lesividad,lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,archivo_origen
0,2018S017842,04/02/2019,9:10:00,"CALL. ALBERTO AGUILERA, 1",1,1,CENTRO,Colisión lateral,Despejado,Motocicleta > 125cc,Conductor,De 45 a 49 años,Hombre,7,Asistencia sanitaria sólo en el lugar del acci...,440068,4475679,N,NaN,300228-11-accidentes-trafico-detalle-csv.csv
1,2018S017842,04/02/2019,9:10:00,"CALL. ALBERTO AGUILERA, 1",1,1,CENTRO,Colisión lateral,Despejado,Turismo,Conductor,De 30 a 34 años,Mujer,7,Asistencia sanitaria sólo en el lugar del acci...,440068,4475679,N,NaN,300228-11-accidentes-trafico-detalle-csv.csv
2,2019S000001,01/01/2019,3:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,11,CARABANCHEL,Alcance,NaN,Furgoneta,Conductor,De 40 a 44 años,Hombre,NaN,NaN,439139,4470836,S,NaN,300228-11-accidentes-trafico-detalle-csv.csv
3,2019S000001,01/01/2019,3:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,11,CARABANCHEL,Alcance,NaN,Turismo,Conductor,De 40 a 44 años,Mujer,NaN,NaN,439139,4470836,N,NaN,300228-11-accidentes-trafico-detalle-csv.csv
4,2019S000001,01/01/2019,3:45:00,PASEO. SANTA MARIA DE LA CABEZA / PLAZA. ELIPTICA,168,11,CARABANCHEL,Alcance,NaN,Turismo,Conductor,De 45 a 49 años,Mujer,NaN,NaN,439139,4470836,N,NaN,300228-11-accidentes-trafico-detalle-csv.csv


In [19]:
acc_raw.info()
print("\nNulos (%):")
print((acc_raw.isna().mean()*100).round(2).sort_values(ascending=False).head(15))
print("\nDuplicados exactos:", acc_raw.duplicated().sum())
for c in ["estado_meteorologico", "tipo_vehiculo", "lesividad", "sexo", "distrito"]:
    if c in acc_raw: print(f"\n{c}: {acc_raw[c].nunique()} valores distintos (antes de normalizar)")

<class 'pandas.DataFrame'>
RangeIndex: 271250 entries, 0 to 271249
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   num_expediente        271250 non-null  str  
 1   fecha                 271250 non-null  str  
 2   hora                  271250 non-null  str  
 3   localizacion          271250 non-null  str  
 4   numero                271242 non-null  str  
 5   cod_distrito          271242 non-null  str  
 6   distrito              271242 non-null  str  
 7   tipo_accidente        271241 non-null  str  
 8   estado_meteorologico  241587 non-null  str  
 9   tipo_vehiculo         269912 non-null  str  
 10  tipo_persona          271247 non-null  str  
 11  rango_edad            271250 non-null  str  
 12  sexo                  271250 non-null  str  
 13  cod_lesividad         148686 non-null  str  
 14  lesividad             148686 non-null  str  
 15  coordenada_x_utm      271209 non-null  str  


**Hallazgos de la auditoría**

- **Accidentes:** 6 CSV (2019-2024) con el mismo esquema de 19 columnas, 271.250 filas y 115.640 accidentes distintos (una fila por persona). El encabezado `estado_meteorológico` lleva tilde y se normaliza; la hora viene como texto `H:MM:SS`.
- **Categorías:** los textos mezclan mayúsculas, tildes y erratas (p. ej. «LLuvia intensa»); se normalizan a minúsculas sin tildes.
- **Nulos relevantes:** lesividad 45,2 % (se trata como «desconocido»), estado meteorológico 10,9 % (más un 1,0 % de «se desconoce»), tipo de vehículo 0,5 % y alcohol 0,3 %. El sexo es «desconocido» en un 10,7 %.
- **Droga:** solo aparece el valor `1` (891 positivos); el nulo se interpreta como «no positivo o sin prueba».
- **Filas idénticas:** 11.449; se conservan porque no hay identificador de persona.
- **AEMET:** 12 JSON con 2.192 días; 20 sin temperatura y 91 sin precipitación. La precipitación inapreciable (`Ip`) y la coma decimal se convierten a número, y no hay datos de viento desde mediados de 2020.
- **Unión:** por fecha, sin pérdida ni duplicación de filas.

## 2. Limpieza de la fuente A

In [20]:
acc, log_acc = clean_accidentes(acc_raw)
for k, v in log_acc.items():
    if k != "nulos_pct": print(f"{k}: {v}")
pd.Series(log_acc["nulos_pct"]).head(10)

filas_iniciales: 271250
filas_identicas_conservadas: 11449
fechas_no_parseables: 0
horas_invalidas: 0
codigos_lesividad_sin_mapear: []
filas_finales: 271250


lesividad               45.18
estado_meteorologico    10.94
tipo_vehiculo            0.49
positiva_alcohol         0.35
coordenada_x_utm         0.02
coordenada_y_utm         0.02
cod_distrito             0.00
numero                   0.00
localizacion             0.00
hora                     0.00
dtype: float64

In [21]:
# Revisar categorías tras normalizar
for c in ["estado_meteorologico", "tipo_accidente", "tipo_vehiculo", "rango_edad", "gravedad"]:
    print(f"\n--- {c}"); print(acc[c].value_counts(dropna=False).head(15))


--- estado_meteorologico
estado_meteorologico
despejado         204139
NaN                29663
nublado            16051
lluvia debil       15084
lluvia intensa      3457
se desconoce        2583
nevando              220
granizando            53
Name: count, dtype: int64

--- tipo_accidente
tipo_accidente
colision fronto-lateral         68697
alcance                         64049
colision lateral                40100
choque contra obstaculo fijo    32638
colision multiple               18020
atropello a persona             17830
caida                           16029
colision frontal                 6569
otro                             4618
solo salida de la via            1144
vuelco                           1025
atropello a animal                504
despenamiento                      18
NaN                                 9
Name: count, dtype: int64

--- tipo_vehiculo
tipo_vehiculo
turismo                              187483
motocicleta hasta 125cc               17855
furgoneta    

### Decisiones sobre nulos
- `tipo_vehiculo` / `rango_edad` nulos: se mantienen como `NaN` y se etiquetan «desconocido» solo en visualización (no se imputan: no hay base para inferirlos).
- `cod_lesividad` nulo → 77 (desconocido), y se excluye de los modelos de gravedad.
- Coordenadas a 0 o nulas: `NaN`; se excluyen solo de los mapas.
- `positiva_alcohol`/`positiva_droga`: nulo ≠ negativo; se conserva `NaN`.

In [22]:
acc["tipo_vehiculo"] = acc["tipo_vehiculo"].fillna("desconocido")
acc["rango_edad"] = acc["rango_edad"].fillna("desconocido")
acc.to_csv(ACCIDENTES_CLEAN, index=False)

## 3. Fuente B: AEMET

In [23]:
met_raw = load_aemet(RAW_AEMET)
met_raw.head()

2019-h2.json: 185 filas
2020-h1.json: 183 filas
2020-h2.json: 185 filas
2021-h1.json: 181 filas
2021-h2.json: 184 filas
2022-h1.json: 180 filas
2022-h2.json: 184 filas
2023-h1.json: 181 filas
2023-h2.json: 184 filas
2024-h1.json: 182 filas
2024-h2.json: 184 filas
aemet_2019_h1.json: 181 filas


,fecha,indicativo,nombre,provincia,altitud,tmed,prec,tmin,horatmin,tmax,horatmax,dir,velmedia,racha,horaracha,presmax,horapresmax,presmin,horapresmin,hrmedia,hrmax,horahrmax,hrmin,horahrmin,pintmax,horapintmax
0,2019-06-30,3195,"MADRID, RETIRO",MADRID,667,"31,8","0,0","25,0",05:40,"38,5",14:00,14,"1,7","7,8",00:00,"941,5",Varias,"938,7",19,33,37,22:30,28,00:00,"0,0",NaN
1,2019-07-01,3195,"MADRID, RETIRO",MADRID,667,"30,0","0,0","23,8",05:15,"36,3",14:10,06,"1,9","12,8",20:10,"940,6",Varias,"937,0",Varias,31,NaN,NaN,NaN,NaN,"0,0",NaN
2,2019-07-02,3195,"MADRID, RETIRO",MADRID,667,"29,2","1,6","23,0",Varias,"35,5",14:40,09,"0,6","10,0",22:20,"940,5",08,"935,0",Varias,33,54,22:30,21,18:00,"5,4",21:40
3,2019-07-03,3195,"MADRID, RETIRO",MADRID,667,"28,6","0,0","22,0",05:00,"35,2",13:50,13,"2,5","12,8",21:50,"938,7",10,"935,1",17,NaN,NaN,NaN,NaN,NaN,"0,0",NaN
4,2019-07-04,3195,"MADRID, RETIRO",MADRID,667,"28,2","0,0","23,1",05:30,"33,2",16:00,15,"2,5","10,8",22:20,"940,0",24,"936,0",00,27,36,Varias,22,17:30,"0,0",NaN


In [24]:
meteo, log_met = clean_aemet(met_raw)
for k, v in log_met.items():
    if k != "nulos_pct": print(f"{k}: {v}")
print(pd.Series(log_met["nulos_pct"]))
meteo.describe().T

filas_iniciales: 2194
fechas_duplicadas_eliminadas: 3
prec_nula_o_acum: 90
tmin_mayor_que_tmax: 0
dias_sin_registro: 1
filas_finales: 2191
fecha                       0.00
meteo_tmed                  0.87
meteo_tmin                  0.87
meteo_tmax                  0.87
meteo_prec                  4.11
meteo_prec_inapreciable     0.00
meteo_velmedia             26.97
meteo_racha                27.38
meteo_presmax               1.78
meteo_presmin               1.83
dtype: float64


,count,mean,min,25%,50%,75%,max,std
fecha,2191,2021-12-31 11:12:21.031492352,2019-01-01 00:00:00,2020-07-01 12:00:00,2021-12-31 00:00:00,2023-07-02 12:00:00,2024-12-31 00:00:00,NaN
meteo_tmed,2172.0,16.335313,-3.4,10.0,15.1,22.525,33.4,7.674715
meteo_tmin,2172.0,11.303039,-7.4,5.9,10.5,16.3,26.2,6.645629
meteo_tmax,2172.0,21.367818,0.3,13.7,19.9,28.8,40.7,8.939434
meteo_prec,2101.0,1.163494,0.0,0.0,0.0,0.0,67.7,4.235048
meteo_prec_inapreciable,2191.0,0.000913,0.0,0.0,0.0,0.0,1.0,0.030206
meteo_velmedia,1600.0,2.068812,0.0,1.4,1.9,2.8,6.4,0.997656
meteo_racha,1591.0,8.919233,2.5,6.7,8.6,10.8,21.1,3.154279
meteo_presmax,2152.0,942.869145,919.5,939.5,942.4,946.2,960.4,5.486467
meteo_presmin,2151.0,938.558531,911.3,935.2,938.4,942.2,957.8,6.183757


In [25]:
meteo.to_csv(METEO_CLEAN, index=False)

## 4. Unión por fecha y variables derivadas

In [26]:
df = merge_accidentes_meteo(acc, meteo)
df = add_calendar_features(df)
df = add_weather_features(df)
df = add_accident_level_features(df)
df.shape

Filas: 271,250 -> 271,250 | sin dato meteorológico: 1.01%


(271250, 47)

In [27]:
# Validaciones finales
assert df.shape[0] >= 50_000, "Menos de 50.000 filas"
assert df.shape[1] >= 20, "Menos de 20 columnas"
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
print("Rango de fechas:", df['fecha'].min().date(), "->", df['fecha'].max().date())
print("Accidentes únicos:", df['num_expediente'].nunique())
print("Sin dato meteo (%):", round(df['meteo_tmed'].isna().mean()*100, 2))
df.dtypes

Filas: 271,250 | Columnas: 47
Rango de fechas: 2019-01-01 -> 2024-12-31
Accidentes únicos: 115640
Sin dato meteo (%): 1.01


num_expediente                     string
fecha                      datetime64[ns]
hora                                  str
localizacion                          str
numero                                str
cod_distrito                          str
distrito                              str
tipo_accidente                        str
estado_meteorologico                  str
tipo_vehiculo                         str
tipo_persona                          str
rango_edad                            str
sexo                                  str
cod_lesividad                       int64
lesividad                             str
coordenada_x_utm                    Int64
coordenada_y_utm                    Int64
positiva_alcohol                  float64
positiva_droga                      int64
archivo_origen                        str
hora_num                            Int64
gravedad                              str
meteo_tmed                        Float64
meteo_tmin                        

In [28]:
df.to_csv(FINAL_CSV, index=False)
print("Guardado en", FINAL_CSV)

Guardado en /Users/miguelfernandez/Desktop/proyecto-eda/data/processed/accidentes_meteo_madrid.csv


## 5. Diccionario de datos del dataset final
Generado automáticamente: origen, tipo, nulos y descripción de cada columna.

In [29]:
colsA = ["num_expediente", "fecha", "hora", "localizacion", "numero", "cod_distrito", "distrito", "tipo_accidente",
         "estado_meteorologico", "tipo_vehiculo", "tipo_persona", "rango_edad", "sexo", "cod_lesividad", "lesividad",
         "coordenada_x_utm", "coordenada_y_utm", "positiva_alcohol", "positiva_droga", "archivo_origen"]
desc = {
 "num_expediente": "Identificador del accidente (una fila por persona implicada)",
 "fecha": "Fecha del accidente", "hora": "Hora (texto H:MM:SS)", "localizacion": "Calle o lugar del accidente",
 "numero": "Número del portal o punto", "cod_distrito": "Código del distrito", "distrito": "Distrito de Madrid",
 "tipo_accidente": "Tipo de accidente (colisión, atropello...)", "estado_meteorologico": "Estado meteorológico anotado por la Policía",
 "tipo_vehiculo": "Tipo de vehículo de la persona", "tipo_persona": "Conductor, pasajero o peatón",
 "rango_edad": "Tramo de edad", "sexo": "Sexo de la persona", "cod_lesividad": "Código de lesividad (nulo = 77, desconocido)",
 "lesividad": "Texto de la lesividad", "coordenada_x_utm": "Coordenada X (UTM)", "coordenada_y_utm": "Coordenada Y (UTM)",
 "positiva_alcohol": "1 positivo, 0 negativo, nulo sin dato", "positiva_droga": "1 positivo; 0 no positivo o sin prueba",
 "archivo_origen": "CSV de procedencia de la fila", "hora_num": "Hora entera (0-23)", "gravedad": "Gravedad según el código de lesividad",
 "meteo_tmed": "Temperatura media del día (°C)", "meteo_tmin": "Temperatura mínima del día (°C)", "meteo_tmax": "Temperatura máxima del día (°C)",
 "meteo_prec": "Precipitación diaria (mm; Ip = 0)", "meteo_prec_inapreciable": "1 si la precipitación fue inapreciable (Ip)",
 "meteo_velmedia": "Velocidad media del viento (m/s)", "meteo_racha": "Racha máxima de viento (m/s)",
 "meteo_presmax": "Presión máxima (hPa)", "meteo_presmin": "Presión mínima (hPa)",
 "anio": "Año", "mes": "Mes (1-12)", "dia_semana": "Día de la semana", "es_fin_semana": "1 si sábado o domingo",
 "estacion": "Estación del año", "es_festivo": "1 si festivo (nacional o de la Comunidad de Madrid)",
 "franja_horaria": "Madrugada (0-5), mañana (6-11), tarde (12-19) o noche (20-23)", "lluvia": "1 si precipitación > 0,1 mm",
 "intensidad_lluvia": "Seco, débil, moderada o fuerte", "rango_termico": "Temperatura máxima menos mínima (°C)",
 "categoria_temp": "Frío, fresco, templado o cálido", "n_implicados": "Personas implicadas en el accidente",
 "gravedad_orden": "Gravedad en número para ordenar", "gravedad_max_accidente": "Gravedad máxima del accidente",
 "es_grave": "1 si grave o mortal", "accidente_con_grave": "1 si el accidente tiene alguna persona grave o mortal",
}
origen = lambda c: "B · AEMET" if c.startswith("meteo_") else ("A · Ayuntamiento" if c in colsA else "Derivada")
dic = pd.DataFrame({"columna": df.columns, "origen": [origen(c) for c in df.columns],
                    "tipo": df.dtypes.astype(str).values, "nulos_%": (df.isna().mean() * 100).round(1).values,
                    "descripcion": [desc.get(c, "") for c in df.columns]})
dic.to_csv(FINAL_CSV.parent / "diccionario_datos.csv", index=False)
dic

,columna,origen,tipo,nulos_%,descripcion
0,num_expediente,A · Ayuntamiento,string,0.0,Identificador del accidente (una fila por pers...
1,fecha,A · Ayuntamiento,datetime64[ns],0.0,Fecha del accidente
2,hora,A · Ayuntamiento,str,0.0,Hora (texto H:MM:SS)
3,localizacion,A · Ayuntamiento,str,0.0,Calle o lugar del accidente
4,numero,A · Ayuntamiento,str,0.0,Número del portal o punto
5,cod_distrito,A · Ayuntamiento,str,0.0,Código del distrito
6,distrito,A · Ayuntamiento,str,0.0,Distrito de Madrid
7,tipo_accidente,A · Ayuntamiento,str,0.0,"Tipo de accidente (colisión, atropello...)"
8,estado_meteorologico,A · Ayuntamiento,str,10.9,Estado meteorológico anotado por la Policía
9,tipo_vehiculo,A · Ayuntamiento,str,0.0,Tipo de vehículo de la persona
